## Model Task / I/O

- **Input:**  
  A batch of RGB face images `x` with shape `[B, 3, H, W]`  
  (typically resized/cropped to `224×224` for `timm` backbones).

- **Output:**  
  For each image, the model outputs **40 real-valued logits** `z`  
  with shape `[B, 40]`, one logit per semantic attribute.

- **Post-processing:**  
  - Apply sigmoid to logits:  
    $p = \sigma(z)$.
  - Threshold probabilities (e.g., `p > 0.5`) to produce  
    **40 binary attribute predictions per image**.

- **Training Objective:**  
  Multi-label classification (each image may have multiple attributes true).  
  Use:
  $$\text{BCEWithLogitsLoss}$$
  between the 40 logits and the 40-dimensional binary target vector.

---

## Dataset / Data Loading

- **Dataset:**  
  CelebA (~200k celebrity face images) with **40 binary attribute annotations** per image.  
  Use the official `train / val / test` split provided by CelebA.

- **Each Sample Provides:**
  - `image`: RGB face photo
  - `target`: 40-dimensional binary vector $y \in \{0,1\}^{40}$
- **The 40 Attributes:**:

- [
  ["5_o_Clock_Shadow", "Arched_Eyebrows", "Attractive", "Bags_Under_Eyes", "Bald", "Bangs"],
  ["Big_Lips", "Big_Nose", "Black_Hair", "Blond_Hair", "Blurry", "Brown_Hair"],
  ["Bushy_Eyebrows", "Chubby", "Double_Chin", "Eyeglasses", "Goatee", "Gray_Hair"],
  ["Heavy_Makeup", "High_Cheekbones", "Male", "Mouth_Slightly_Open", "Mustache", "Narrow_Eyes"],
  ["No_Beard", "Oval_Face", "Pale_Skin", "Pointy_Nose", "Receding_Hairline", "Rosy_Cheeks"],
  ["Sideburns", "Smiling", "Straight_Hair", "Wavy_Hair", "Wearing_Earrings", "Wearing_Hat"],
  ["Wearing_Lipstick", "Wearing_Necklace", "Wearing_Necktie", "Young"]
]


## Probe Inspirations (CelebA Multi-Label Attribute Classification Context)
### 1. Region–Attribute Causal Sensitivity Probe
**Description:**  
Evaluate whether an attribute prediction is causally driven by its semantically relevant facial region rather than peripheral cues.
**Example:**  
Mask or occlude the mouth region when predicting *Wearing_Lipstick* or *Smiling*. Compute Δ-logit or Δ-probability before and after masking.  
- Large change when masking the correct region → expected causal dependence.  
- Large change when masking irrelevant regions (e.g., hair, background) → spurious feature reliance.  
This probe directly tests whether the model localizes evidence appropriately.
### 2. Contextual Shortcut Dependency Probe
**Description:**  
Assess whether predictions depend on background, clothing, or illumination rather than intrinsic facial features.
**Example:**  
Apply controlled background blur or replacement while preserving facial pixels. Measure prediction shift for attributes like *Male*, *Young*, or *Smiling*.  
Significant Δ-probability under background-only edits suggests shortcut learning from contextual co-occurrence patterns.
### 3. Cross-Attribute Logical Consistency Probe
**Description:**  
Examine whether the model encodes coherent relationships across attributes instead of predicting them independently.
**Example:**  
Quantify violation rate of logically incompatible pairs (e.g., *Bald* ∧ *Bangs*, *No_Beard* ∧ *Mustache*).  
High joint probability for inconsistent attribute pairs indicates weak structural modeling and potential reliance on local texture statistics rather than global facial semantics.
### 4. Demographic Proxy Sensitivity Probe
**Description:**  
Test whether non-visual demographic attributes disproportionately influence cosmetic or appearance-based predictions.
**Example:**  
Analyze conditional probabilities such as  
P(Wearing_Lipstick | Male = high) vs P(Wearing_Lipstick | Male = low).  
If lipstick prediction systematically shifts with predicted *Male* status even when visual evidence is constant, the model may encode demographic bias instead of visual causation.
### 5. Controlled Perturbation Stability Probe
**Description:**  
Measure robustness under non-semantic perturbations and responsiveness under semantic edits.
**Example:**  
Apply mild brightness, color jitter, or Gaussian noise.  
- Excessive prediction variance under minor pixel changes → over-sensitivity to low-level statistics.  
- Minimal response to meaningful edits (e.g., synthetic hair recoloring for *Blond_Hair*) → failure to encode true semantic feature.  

Once you walk through the examples I just gave you above, you should, based on our task description and dataset head, give five probe designs based on our example and another five probe designs that do not come from our example. Totally there should be ten probes to return.

[PROBE_DESIGN_REQUIREMENT]: For each probe you give, you should always provide the following:

1. Probe type and probe name  
2. Explanation of why you choose to design this probe  
   - When giving explanations, you must ensure they are highly relevant and critically connected to our task topic  
3. If possible, try to provide a reference that supports your design (OPTIONAL)  
4. How much confidence you have in this probe (ranged by [0,1])  

Always return in JSON format such that it is easy and ready to extract your response. Also, always return the probe designs ranked by the confidence (mentioned in [PROBE_DESIGN_REQUIREMENT].4) in descending order.

Below is a JSON template for organizing your response:

```json
{
  "probe_designs": [
    {
      "probe_type": "string",
      "probe_name": "string",
      "explanation": "string",
      "reference": "string or null",
      "confidence": "float between 0 and 1"
    }
    // ... rest probes
  ]
}

We will proceed with this probe idea:
### Role Setup
Imagine you are a professional Machine Learning Engineer (MLE). Read the following file relationships carefully:
- You are allowed to read `./` for information.
- `.wrapper_plot.py` — contains plotting functions you must use to display your probe conclusion (see head comment for details).
### Probe Intention
The probe.py should reflect the quality of the current training process.
For example, you may extract statistics such as true/false rate, prediction error, etc., then:
- Construct probe logic based on these data.
- Generate a conclusion.
- Present the conclusion using plots (via `wrapper_plot.py`).


## Core Implementation Principle
Your probe must explicitly define:
### 1. Metric
A numerical measurement that quantifies model performance.
- The metric **must be computed from data accessible during training**.
- It must be clearly defined with a precise computation formula.
- Example: covariance between model quality and race label in a bias probe.
---
### 2. Expectation (a string)
A description of what the metric should look like for a well-trained model.
- Define what “good performance” means numerically.
- Clearly interpret what values indicate strong vs weak training.
- Example: A well-trained model should produce balanced subgroup distributions.
---
### 3. Threshold (a numerical value)
A numerical decision boundary used to judge performance.
- Explicitly define the threshold.
- Compare metric value against threshold.
- Example: If metric > 0.5 and threshold = 0.5 → considered good.
You must clearly define:
- Metric
- Expectation
- Threshold
- Their logical relationship
### 4. visualization
you must read and properly call wrapper_plot.py to display your conclusion as figure/plot
### 5. other Required
When designing the probe:
- Explicitly state metric, expectation, and threshold.
- Clearly describe:
  - Data source
  - Data processing steps
  - Computation formula
  - Interpretation logic
- Use functions from `wrapper_plot.py` for visualization. read what plot is supported in wrapper_plot and consider how to design your implementation
- stdout your probe conclusions in a JSON file in the following format:

```json
{
  "metric_data1": "numerical value",
  "threshold": "numerical value",
  "expectation": "textual interpretation of how to evaluate the metric"
}
```


## Your Task
Generate **three different implementation (`dev_docs`)**.
### Requirements for Each `dev_doc`
Each `dev_doc` must:
- Follow a **consistent and structured format**
- Clearly define:
  - **Metric**
  - **Expectation**
  - **Threshold**
- Describe:
  - **Data collection method**
  - **Computation formula**
  - **Result interpretation logic**
  - **Visualization plan**
  - **JSON output structure**
- Be written **clearly, logically, and concisely**
- Only use **data accessible during training**
### Output Requirements
return a json in following schema to me in the conversation
### JSON Schema Requirement
Your output must follow this exact structure:
```json
{
  "dev_docs": [
    {
      "rank": 1,
      "title": "string",
      "confidence": 0.0,
      "metric": {
        "name": "string",
        "definition": "string",
        "formula": "string"
      },
      "expectation": "string",
      "threshold": {
        "value": 0.0,
        "comparison_logic": "string"
      },
      "data_source": "string",
      "data_processing_steps": ["step1", "step2"],
      "interpretation_logic": "string",
      "visualization_plan": "string (must reference wrapper_plot functions)",
      "json_output_example": {
        "metric_name": 0.0,
        "threshold": 0.0,
        "expectation": "string",
        "interpretation": "string"
      }
    }
  ]
}